# Run 6 Xe Fast Analysis 教学版

这个 notebook 是根据 `run6_xe_fast_0611.ipynb` 前半部分整理出来的教学流程。目标不是一次性给出最终 cut，而是按数据处理链路说明：

1. 如何扫描 run 并选择 `run_id`。
2. 如何建立 `Context`、注册插件、设置关键配置。
3. 如何读取 `records` 和连续波形池 `wave_pool`。
4. 如何做 records、hits、peaks 三个层级的基础诊断。

建议从上到下顺序运行。每一节的 markdown 解释“这一步在检查什么”，代码 cell 尽量保持可以单独复用。


# 设置配置

## 1. 初始化环境

`autoreload` 让本地 Python 包修改后可以在 notebook 中自动刷新。下面同时导入后续各节共用的科学计算和绘图工具。


In [ ]:
%load_ext autoreload
%autoreload 2

import os

from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 120

## 1. 扫描 run 列表

这一节用于确认数据目录里有哪些 run，以及它们的采集时间。`max_waves_for_channels` 控制扫描时每个通道最多读取多少条波形，教学时可以先保持较小，减少等待时间。


In [ ]:
from waveform_analysis import DAQAnalyzer

DATA_ROOT = "/mnt/data/TPC/run6_Xe/"
RUN_CONFIG_PATH = "/mnt/data/TPC/run6_Xe/run_config.json"
CUSTOM_CONFIG_PATH = "run6_xe_fast_0611.json"

analyzer = DAQAnalyzer(
    DATA_ROOT,
    daq_adapter="v1725",
    max_waves_for_channels=1000,
)

analyzer.scan_all_runs(
    start_time="2026-06-10",
    end_time="2026-06-16",
)


In [ ]:
analyzer.display_overview(
    sort_by="time",
    ascending=False,
    max_rows=100,
)

## 2. 建立 Context 并注册插件

`Context` 是这套分析框架的入口：

- `ctx.register(...)` 注册数据读取、波形处理、特征提取、peak 构建等插件。
- `ctx.get_data(run_id, target)` 会按插件依赖关系自动计算或读取结果。
- `ctx.plot_lineage(target)` 可以查看某个 target 依赖哪些上游数据。

这里先固定一个教学用 `run_id`。如果你要换 run，只需要改下一格的 `run_id`。


In [ ]:
from waveform_analysis.core.context import Context
from waveform_analysis.core.plugins.plugin_sets import (
    plugins_basic_features,
    plugins_events,
    plugins_io,
    plugins_peaks,
    plugins_tabular,
    plugins_waveform,
)

run_id = "00195"
ctx = Context(storage_dir=DATA_ROOT)

ctx.register(*plugins_io())
ctx.register(*plugins_waveform())
ctx.register(*plugins_basic_features())
ctx.register(*plugins_peaks())
ctx.register(*plugins_events())
ctx.register(*plugins_tabular())


## 3. 设置处理参数

下面这组配置基本对应原 notebook 的前半部分。教学时重点关注几个参数：

- `daq_adapter="v1725"`：匹配当前数据文件格式。
- `hit_threshold.threshold`：单通道 hit 的阈值。
- `hit_merged.merge_gap_ns`：相邻 hit 合并成一个 merged hit 的最大间隔。
- `peaklets.time_window_ns` 和 `max_total_width_ns`：merged hit 继续聚合成 peaklet 的时间窗口。

如果调整这些参数，后续 records/hits/peaks 的分布会随之改变。


In [ ]:
ctx.set_config({
    "data_root": DATA_ROOT,
    "run_config_path": RUN_CONFIG_PATH,
    "custom_config_json_path": CUSTOM_CONFIG_PATH,
    "daq_adapter": "v1725",
    "show_progress": True,
    "use_filtered": False,
    "wave_source": "records",
})

ctx.set_config(
    {
        "channel_workers": 14,
        "n_jobs": 14,
        "v1725_part_size": 20_000,
        "use_process_pool": True,
        "channel_executor": "process",
        "keep_on_disk": True,
    },
    plugin_name="records",
)

ctx.set_config({"area_range": (0, None), "height_range": (0, None)}, "basic_features")

ctx.set_config(
    {
        "asymmetry_cut_min": 0.7,
        "asymmetry_chunk_size": 200_000,
        "asymmetry_num_threads": 128,
    },
    "records_asymmetry_mask",
)

ctx.set_config(
    {
        "threshold": 100,
        "asymmetry_cut_enabled": True,
        "right_extension": 5,
    },
    plugin_name="hit_threshold",
)

ctx.set_config({"merge_gap_ns": 400, "max_total_width_ns": 40000}, "hit_merged")

ctx.set_config(
    {
        "time_window_ns": 400.0,
        "max_total_width_ns": 50000.0,
        "dt": 4,
    },
    "peaklets",
)


In [ ]:
ctx.show_config("hit_threshold")


In [ ]:
ctx.plot_lineage("basic_features", verbose=1)


# Records 波形的读取


## 4. 读取 raw files、records 和 wave_pool

`records` 是每条触发波形的元数据表，真正的 ADC 波形存放在连续数组 `wave_pool` 里。`records[i]` 里的 `wave_offset` 和 `event_length` 可以定位第 `i` 条波形。

这里使用 `records_view` 包装读取逻辑，后续画波形时会更方便。


In [ ]:
raw_files = ctx.get_data(run_id, "raw_files")
raw_files


In [ ]:
from waveform_analysis.core import records_view

rv = records_view(ctx, run_id)
records = rv.records
wave_pool = rv.wave_pool


def get_wave(i: int):
    """Return waveform samples for the i-th row in records."""
    off = int(records["wave_offset"][i])
    n = int(records["event_length"][i])
    return wave_pool[off : off + n].view("i2")

print(f"records: {len(records):,}")
print(f"wave_pool samples: {len(wave_pool):,}")


In [ ]:
pd.DataFrame(records[:10])


## 5. 预览各通道波形

这一格按 board/channel 分组，画出每组前 20 条记录的波形。它的作用是快速发现明显问题：通道是否有数据、baseline 是否异常、是否存在非常规饱和或极性问题。


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 6))
axes = axes.flatten()

plot_index = 0
for board in range(0, 2):
    for channel in range(12, 16):
        ax = axes[plot_index]
        select_record_id = records[(records["channel"] == channel) & (records["board"] == board)]["record_id"]
        if len(select_record_id) > 0:
            ax.plot(rv.signals(select_record_id[:20]).T, linewidth=0.8, alpha=0.7)
        ax.set_title(f"Board {board}, Channel {channel}")
        ax.set_xlabel("sample")
        ax.set_ylabel("ADC")
        plot_index += 1

for ax in axes[plot_index:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Records level: area-height 分布

`df = ctx.get_data(run_id, "df")` 通常是 records 层级基础特征的表格化结果。下面用 `area` 和 `height` 的二维直方图检查每个通道的分布。

红色虚线是一个教学用的经验分界：它把每个 record 按 `height` 相对 `area` 的位置分成 `above / middle / below` 三类。这个分类不是最终物理选择，只是帮助理解分布结构。


In [ ]:
df = ctx.get_data(run_id, "df")

fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex=True, sharey=True)

x_bins = np.logspace(1, 7, 200)
y_bins = np.logspace(2, 4.5, 200)
norm = LogNorm(vmin=1, vmax=10000)

k_low, b_low = 0.8, -0.4
k_high, b_high = 1.0, -0.4
area_min = 2e2
area_max = 2e7

x_line = np.logspace(np.log10(area_min), np.log10(area_max), 200)
y_low = 10 ** (k_low * np.log10(x_line) + b_low)
y_high = 10 ** (k_high * np.log10(x_line) + b_high)

mask_above = pd.Series(False, index=df.index)
mask_middle = pd.Series(False, index=df.index)
mask_below = pd.Series(False, index=df.index)

for board in [0, 1]:
    for ch in range(12, 16):
        ax = axes[ch - 12, board]
        df_ch = df[(df["channel"] == ch) & (df["board"] == board)]

        hist = ax.hist2d(
            df_ch["area"],
            df_ch["height"],
            bins=(x_bins, y_bins),
            norm=norm,
            cmin=1,
        )
        im = hist[3]

        area = df_ch["area"]
        height = df_ch["height"]
        valid = (area > 0) & (height > 0) & (area > area_min) & (area < area_max)

        loga = np.log10(area[valid])
        logh = np.log10(height[valid])
        line_low = k_low * loga + b_low
        line_high = k_high * loga + b_high
        lower = np.minimum(line_low, line_high)
        upper = np.maximum(line_low, line_high)

        idx = logh.index
        mask_above.loc[idx] = logh > upper
        mask_middle.loc[idx] = (logh >= lower) & (logh <= upper)
        mask_below.loc[idx] = logh < lower

        ax.plot(x_line, y_low, "r--", lw=1.2)
        ax.plot(x_line, y_high, "r--", lw=1.2)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(1e1, 1e7)
        ax.set_ylim(1e2, np.nanmax(df["height"]))
        ax.set_xlabel("area")
        ax.set_ylabel("height")
        ax.set_title(f"board {board}, channel {ch}")
        fig.colorbar(im, ax=ax, label="counts")

fig.tight_layout()
plt.show()

df_above = df[mask_above].copy()
df_middle = df[mask_middle].copy()
df_below = df[mask_below].copy()

summary = pd.DataFrame(
    {
        "count": [len(df_above), len(df_middle), len(df_below)],
        "fraction": [len(df_above) / len(df), len(df_middle) / len(df), len(df_below) / len(df)],
    },
    index=["above", "middle", "below"],
)
summary


## 7. Records level: 时间间隔和波形长度

records 的相邻时间间隔能帮助判断触发间隔、噪声簇和异常密集区域。`event_length` 分布则用于检查记录长度是否如预期，例如是否有大量异常长波形。


In [ ]:
pairs = np.unique(np.c_[records["board"], records["channel"]], axis=0)
pairs = pairs[np.lexsort((pairs[:, 1], pairs[:, 0]))]
bins = np.logspace(1, 5.5, 200)

for bd, ch in pairs:
    mask = (records["board"] == bd) & (records["channel"] == ch)
    ts = records["timestamp"][mask]

    fig, ax = plt.subplots(figsize=(6, 4))
    if len(ts) < 2:
        ax.text(0.5, 0.5, f"board {int(bd)}, ch {int(ch)}\nno data", ha="center", va="center")
        ax.axis("off")
    else:
        diffs_ns = np.diff(ts) / 1e3
        ax.hist(diffs_ns, bins=bins, log=True)
        ax.set_xscale("log")
        ax.set_title(f"board {int(bd)}, ch {int(ch)} (n={len(diffs_ns)})")
        ax.set_xlabel("Time difference [ns]")
        ax.set_ylabel("counts")
    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(records["event_length"], bins=np.linspace(0, 10000, 200))
plt.yscale("log")
plt.xlabel("event_length [samples]")
plt.ylabel("counts")
plt.title("Record waveform length")
plt.show()


# Hit level 数据


## 8. Hit level: threshold hits 和 merged hits

`hit_threshold` 是单通道超过阈值的 hit。`hit_merged` 是把时间上接近的 hit 合并后的结果。这里先看同一通道内相邻 hit 的时间差，再看 merged hit 的表格结构。


In [ ]:
hit_threshold = ctx.get_data(run_id, "hit_threshold", output="array")
hit_merged = ctx.get_data(run_id, "hit_merged", output="array")
hit_merged_components = ctx.get_data(run_id, "hit_merged_components", output="array")

print(f"hit_threshold: {len(hit_threshold):,}")
print(f"hit_merged: {len(hit_merged):,}")
print(f"hit_merged_components: {len(hit_merged_components):,}")
pd.DataFrame(hit_merged[:10])


In [ ]:
channel = 14
h = hit_threshold[hit_threshold["channel"] == channel]

order = np.lexsort((h["timestamp"], h["record_id"]))
h = h[order]
same_record = h["record_id"][1:] == h["record_id"][:-1]

dt_all_ns = np.diff(h["timestamp"].astype(np.int64)) / 1e3
dt_same_record_ns = np.diff(h["timestamp"].astype(np.int64))[same_record] / 1e3

plt.figure(figsize=(7, 4))
plt.hist(dt_all_ns, bins=np.linspace(0, 2000, 500), histtype="step", label="All hits")
plt.hist(dt_same_record_ns, bins=np.linspace(0, 2000, 500), histtype="step", label="Same record")
plt.yscale("log")
plt.xlabel("time difference [ns]")
plt.ylabel("counts")
plt.title(f"channel {channel}, hit_threshold time difference")
plt.ylim(10, None)
plt.axvline(400, color="red", linestyle="--", label="merge_gap_ns=400")
plt.legend()
plt.show()


下面的函数使用 hit 的窗口边界计算 merged hit 内部相邻 hit 的间隔。它比只比较 `timestamp` 更接近“两个 hit 波形窗口之间还有多少空隙”。


In [ ]:
def _has_field(arr, name):
    if hasattr(arr, "columns"):
        return name in arr.columns
    names = getattr(getattr(arr, "dtype", None), "names", None)
    return names is not None and name in names


def get_all_hit_intervals_fast(
    hit_merged_components,
    hit_threshold,
    *,
    same_channel=True,
    divisor=1e3,
):
    merged_index = np.asarray(hit_merged_components["merged_index"], dtype=np.int64)
    hit_index = np.asarray(hit_merged_components["hit_index"], dtype=np.int64)

    timestamp = np.asarray(hit_threshold["timestamp"][hit_index], dtype=np.int64)
    position = np.asarray(hit_threshold["position"][hit_index], dtype=np.int64)
    board = np.asarray(hit_threshold["board"][hit_index], dtype=np.int64)
    channel = np.asarray(hit_threshold["channel"][hit_index], dtype=np.int64)

    if _has_field(hit_threshold, "edge_start"):
        edge_start = np.asarray(hit_threshold["edge_start"][hit_index], dtype=np.int64)
        edge_end = np.asarray(hit_threshold["edge_end"][hit_index], dtype=np.int64)
    else:
        edge_start = np.asarray(hit_threshold["sample_start"][hit_index], dtype=np.int64)
        edge_end = np.asarray(hit_threshold["sample_end"][hit_index], dtype=np.int64)

    dt_ns = np.asarray(hit_threshold["dt"][hit_index], dtype=np.int64)
    dt_ps = dt_ns * 1000

    abs_start_ps = timestamp + (edge_start - position) * dt_ps
    abs_end_ps = timestamp + (edge_end - position) * dt_ps

    if same_channel:
        order = np.lexsort((abs_start_ps, channel, board, merged_index))
    else:
        order = np.lexsort((abs_start_ps, merged_index))

    merged_index = merged_index[order]
    abs_start_ps = abs_start_ps[order]
    abs_end_ps = abs_end_ps[order]
    board = board[order]
    channel = channel[order]
    dt_ns = dt_ns[order]

    gap_ps = abs_start_ps[1:] - abs_end_ps[:-1]
    dt = gap_ps / divisor

    same_group = (merged_index[1:] == merged_index[:-1]) & (dt_ns[1:] == dt_ns[:-1])
    if same_channel:
        same_group &= (board[1:] == board[:-1]) & (channel[1:] == channel[:-1])

    return dt[same_group]


dt_same_channel = get_all_hit_intervals_fast(
    hit_merged_components,
    hit_threshold,
    same_channel=True,
)

plt.figure(figsize=(8, 5))
plt.hist(dt_same_channel, bins=np.linspace(0, 5e3, 500), histtype="step")
plt.yscale("log")
plt.xlabel("Adjacent hit window gap within hit_merged [ns]")
plt.ylabel("Counts")
plt.title("Hit intervals using window boundaries")
plt.grid(True, alpha=0.3)
plt.show()

pd.Series(dt_same_channel).describe()


# Peaks level

## 9. Peaks level: 读取 peaks 并查看参数空间

`peaks` 是更高层级的候选信号结构。这里先看 lineage，再在多个变量之间做 corner plot：

- `area` 和 `height` 描述总电荷和峰高。
- `width_25_75`、`rise_time_10_50`、`fall_time` 描述形状。
- `n_hits` 和 `n_channels` 描述有多少 hit/通道参与。

教学版保留一个简单筛选 `peaks_filtered`，用于后续看典型波形。

In [ ]:
from waveform_analysis.utils import corner_hist, plot_1d_cut_on_corner

peaks_raw = ctx.get_data(run_id, "peaks", output="array")
print(f"peaks: {len(peaks_raw):,}")
ctx.plot_lineage("peaks", verbose=2)

In [ ]:
from waveform_analysis.utils import plot_1d_cut_on_corner

bins = (
    np.logspace(2, 7, 200),
    np.logspace(2, 5, 200),
    np.logspace(0, np.log10(1e4), 200),
    np.logspace(0, np.log10(5e3), 200),
    np.logspace(0, np.log10(5e3), 100),
    np.linspace(0, 100, 100),
    np.linspace(0, 10, 11),
)

names = ["area", "height", "width", "rise_time_10_50", "fall_time", "n_hits", "n_channels"]

fig, axes = corner_hist(
    (
        peaks_raw["area"],
        peaks_raw["height"],
        peaks_raw["width_25_75"],
        peaks_raw["rise_time_10_50"],
        peaks_raw["fall_time"],
        peaks_raw["n_hits"],
        peaks_raw["n_channels"],
    ),
    names=names,
    bins=bins,
    scales=("log", "log", "log", "log", "log", "linear", "linear"),
    triangle="lower",
    label_mode="outer",
    hist2d_alpha=0.3,
)

mask = (
    (peaks_raw["n_channels"] < 8)
    & (peaks_raw["area"] > 7e3)
    & (peaks_raw["width_25_75"] > 90)
    & (peaks_raw["width_25_75"] < 250)
)
peaks_filtered = peaks_raw[mask]

corner_hist(
    (
        peaks_filtered["area"],
        peaks_filtered["height"],
        peaks_filtered["width_25_75"],
        peaks_filtered["rise_time_10_50"],
        peaks_filtered["fall_time"],
        peaks_filtered["n_hits"],
        peaks_filtered["n_channels"],
    ),
    names=names,
    bins=bins,
    scales=("log", "log", "log", "log", "log", "linear", "linear"),
    triangle="lower",
    label_mode="outer",
    hist_color="C2",
    fig=fig,
    axes=axes,
    show_ticks=True,
)

plot_1d_cut_on_corner(axes, names, var="width", value=90, triangle="lower")
plot_1d_cut_on_corner(axes, names, var="width", value=250, triangle="lower")
plot_1d_cut_on_corner(axes, names, var="area", value=7e3, triangle="lower")
plt.show()

print(f"selected peaks: {len(peaks_filtered):,} / {len(peaks_raw):,}")


## 10. 查看 peak 对应的多通道波形

最后定义一个教学用的波形检查函数。它从 `peaklet_components -> hit_merged -> records/wave_pool` 追溯到原始波形窗口，并把所有通道的波形和求和波形画出来。

这一步的意义是把参数空间里的一个点，重新对应回真实波形，检查 cut 是否选到了预期形状。


In [ ]:
from waveform_analysis.utils.peak_channel_accessor import PeakChannelAccessor

# 1. 创建访问器
accessor = PeakChannelAccessor(ctx, run_id)


`channels = accessor.get_peak_channel_data(peak_id=42, include_waveform=True)`  
这一步会返回这个 `peak_id` 对应的所有通道信息，每个元素都是一个字典。

每个 `ch` 通常包含两类内容：

- **特征值**：`area`、`height`、`width`、`n_hits`、`n_channels` 等
- **波形信息**：`waveform`、`time_ns`、`signal` 等

所以这里的 `channels` 可以理解成：

> “这个 peak 在每个通道上的特征 + 对应波形窗口”

这样做的好处是，前面可以先用 `get_peak_channels()` 快速看特征，  
需要进一步检查形状时，再用 `include_waveform=True` 把原始波形一起取出来。

下面的循环里：

```python
for ch in channels:
    print(f"Waveform shape: {ch['waveform'].shape}")

In [ ]:
# 2. 获取通道特征（快速，不加载波形）
channels = accessor.get_peak_channels(peak_id=919)
for ch in channels:
    print(f"Channel {ch['channel']}: area={ch['area']:.1f}")

In [ ]:
# 3. 获取特征 + 波形
channels = accessor.get_peak_channel_data(peak_id=919, include_waveform=True)

channels

In [ ]:

# 3. 获取特征 + 波形
channels = accessor.get_peak_channel_data(peak_id=42, include_waveform=True)
for ch in channels:
    print(f"Waveform shape: {ch['waveform'].shape}")

# 4. 可视化
accessor.plot(peak_id=42)  # 绘制单个 peak
accessor.batch_plot([42, 43, 44])  # 批量绘制

In [ ]:
peak_id = int(peaks_filtered["peak_id"][0])
channels = accessor.get_peak_channel_data(peak_id=peak_id, include_waveform=True)
for ch in channels:
    print(
        f"peak_id={peak_id}, channel={ch['channel']}, area={ch['area']:.1f}, "
        f"waveform_shape={ch['waveform'].shape}"
    )

In [ ]:
for peak_id in peaks_filtered["peak_id"][:4]:
    peak_id = int(peak_id)
    channels = accessor.get_peak_channels(peak_id=peak_id)
    print(
        f"peak_id={peak_id}, n_channels={len(channels)}, "
        f"areas={[round(ch['area'], 1) for ch in channels[:5]]}"
    )
    fig, axes = accessor.plot(peak_id=peak_id)
    plt.show()

## 11. 下一步可以怎么扩展

完成这个教学流程后，可以继续接原 notebook 后半部分：

- 研究 `rise_time`、`width` 等参数的物理含义。
- 设计 S1/S2 或单电子候选的 cut。
- 建立 S1-S2 配对并检查配对后的参数空间。
- 把稳定下来的诊断函数移动到 `src/` 或 `utils/`，方便测试和复用。
